In [ ]:
import pandas as pd

In [4]:
import pandas as pd
df = pd.read_csv("train.csv")
print(df.columns.tolist())
df.head()

['project_id', 'num_commits', 'active_commit_days', 'project_span_days', 'commit_cadence', 'pct_commits_weekend', 'pct_commits_late_night', 'lines_added_total', 'lines_deleted_total', 'churn_ratio', 'median_lines_per_commit', 'max_lines_in_one_commit', 'pct_code_in_first_commit', 'num_commit_authors', 'avg_commit_msg_length', 'pct_generic_commit_msgs', 'num_branches', 'num_merge_commits', 'lines_of_code', 'num_source_files', 'num_directories', 'max_dir_depth', 'num_languages', 'primary_language', 'pct_primary_language', 'num_dependencies', 'framework', 'uses_game_engine', 'has_tests', 'num_test_files', 'has_ci_config', 'has_dockerfile', 'has_backend', 'has_database', 'avg_file_length', 'comment_line_ratio', 'num_image_files', 'num_audio_files', 'num_3d_model_files', 'num_font_files', 'asset_size_mb', 'has_custom_art', 'readme_word_count', 'readme_num_headings', 'readme_num_images', 'readme_num_code_blocks', 'readme_has_install_instructions', 'has_license', 'has_docs_folder', 'url_is_li

,project_id,num_commits,active_commit_days,project_span_days,commit_cadence,pct_commits_weekend,pct_commits_late_night,lines_added_total,lines_deleted_total,churn_ratio,...,forks,num_issues,num_pull_requests,num_releases,is_forked_from_template,author_prior_projects,author_account_age_years,hours_spent,log_hours,hours_bucket
0,535ce026f7,44,15,56,daily,0.2045,0.4545,8500,2900,0.3438,...,0,0,0,0,False,0,1-2,32.2,3.4726,30-100
1,869ba41ed7,9,3,76,sporadic,0.8889,0.0000,500,100,0.2484,...,0,0,0,0,False,0,3-5,5.0,1.6094,<10
2,d6f81f8862,12,1,0,frequent,1.0000,0.0000,5900,100,0.0247,...,0,0,0,0,False,0,3-5,1.3,0.2624,<10
3,152b5d915d,4,1,0,frequent,0.0000,0.2500,6000,0,0.0000,...,0,0,0,0,False,0,<1,2.0,0.6931,<10
4,3b5f8a49e6,7,3,64,sporadic,0.8571,0.0000,800,0,0.0419,...,0,0,0,0,False,0,1-2,4.0,1.3863,<10


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import root_mean_squared_log_error
import numpy as np
import lightbgm as lgb

drop_cols = ["project_id", "hours_spent", "log_hours", "hours_bucket"]
features = [c for c in train.columns if c not in drop_cols]
X = df[features].copy()
for col in features:
    if X[col].dtype == "object":
        X[col] = X[col].astype("category")

X_tr, X_va, y_tr, y_va, h_tr, h_va = train_test_split(
    X, df["log_hours"], df["hours_spent"], test_size=0.2, random_state=42
)
m = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=31, random_state=42)
m.fit(X_tr, y_tr)
preds = np.expm1(m.predict(X_va)).clip(0.1, 140.7)
print("Local RMSLE:", root_mean_squared_log_error(h_va, preds))